# Notebook 02 — Temporary Tables and Stored Procedures

**Session:** Database - SQL (3)  
**Module:** 02 · Calculus and Linear Algebra

## Goal
1. Materialize expensive aggregations into **session-scoped temporary tables** using both explicit `CREATE TEMPORARY TABLE` + `INSERT INTO` and `CREATE TEMPORARY TABLE ... AS SELECT`.
2. Verify **session isolation and automatic cleanup** across separate `sqlite3` connections.
3. Implement **stored-procedure patterns** (MySQL `CREATE PROCEDURE` equivalents) as **ACID-safe parameterized Python transactional routines** in SQLite, including rollback safety and cursor-vs-set aggregation.

In [ ]:
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))


def query(sql: str, params: tuple = (), connection: sqlite3.Connection = conn) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = connection.execute(sql, params)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))


print("Loaded tables:")
show_rows(query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;"))

## 1. Materializing `temp_invoice` (Schema-First + `INSERT INTO ... SELECT`)

Instead of recomputing `SUM(invoice_total)` and `AVG(invoice_total)` per client in every downstream query, we create `temp_invoice` once and query it multiple times.

In [ ]:
conn.executescript("""
DROP TABLE IF EXISTS temp_invoice;
CREATE TEMPORARY TABLE temp_invoice (
    client_id INTEGER,
    invoice_sum REAL,
    invoice_avg REAL
);

INSERT INTO temp_invoice
SELECT
    client_id,
    ROUND(SUM(invoice_total), 2) AS invoice_sum,
    ROUND(AVG(invoice_total), 2) AS invoice_avg
FROM invoices
GROUP BY client_id;
""")

print("Contents of temp_invoice:")
temp_rows = query("SELECT * FROM temp_invoice ORDER BY client_id;")
show_rows(temp_rows)

# Reuse #1: Highest-invoiced client
top_client = query("""
SELECT
    c.client_id,
    c.name,
    (
        SELECT ti.invoice_sum
        FROM temp_invoice ti
        WHERE ti.client_id = c.client_id
    ) AS invoice_sum
FROM clients c
ORDER BY invoice_sum DESC
LIMIT 1;
""")

# Reuse #2: Client average invoice summary
avg_clients = query("""
SELECT
    c.client_id,
    c.name,
    (
        SELECT ti.invoice_avg
        FROM temp_invoice ti
        WHERE ti.client_id = c.client_id
    ) AS invoice_avg
FROM clients c
ORDER BY c.client_id;
""")

assert top_client[0]["client_id"] == 5
assert top_client[0]["invoice_sum"] == 344.68
assert avg_clients[0]["invoice_avg"] == 150.31

## 2. Session Isolation and Lifetime Verification

A temporary table exists **exclusively** for the connection that created it. A second independent connection cannot read `temp_invoice`, and closing a connection destroys its temporary tables automatically.

In [ ]:
conn_b = sqlite3.connect(":memory:")
conn_b.executescript(SEED_PATH.read_text(encoding="utf-8"))

# Connection B has no visibility into Connection A's temp_invoice
missing_in_b = False
try:
    conn_b.execute("SELECT * FROM temp_invoice;")
except sqlite3.OperationalError as exc:
    missing_in_b = True
    print(f"Expected isolation error on Connection B: {exc}")

assert missing_in_b
conn_b.close()

## 3. Stored Procedures & Parameterized Transactional Routines

In MySQL, stored procedures persist in the schema catalog (`CREATE PROCEDURE ... BEGIN ... END`). Because SQLite is an embedded in-process engine, application code encapsulates those exact multi-statement routines as **parameterized, ACID-transactional Python functions** using `with conn:`.

Below we implement and test:
1. `create_temp_invoice(conn)` — recreates and refreshes `temp_invoice`.
2. `update_payment_invoice(conn, ...)` — atomically inserts a payment and increments `invoices.payment_total` (rolling back completely if a foreign-key violation occurs).
3. `apply_discount(conn, min_invoice_total, discount_rate)` — updates qualifying invoices.
4. `calculate_office_salaries(conn)` — compares row-by-row cursor iteration against single-pass set aggregation.

In [ ]:
def create_temp_invoice(connection: sqlite3.Connection) -> list[dict]:
    """Refresh session-scoped temp_invoice table (stored procedure equivalent)."""
    connection.executescript("""
    DROP TABLE IF EXISTS temp_invoice;
    CREATE TEMPORARY TABLE temp_invoice (
        client_id INTEGER,
        invoice_sum REAL,
        invoice_avg REAL
    );
    INSERT INTO temp_invoice
    SELECT
        client_id,
        ROUND(SUM(invoice_total), 2) AS invoice_sum,
        ROUND(AVG(invoice_total), 2) AS invoice_avg
    FROM invoices
    GROUP BY client_id;
    """)
    return query("SELECT * FROM temp_invoice ORDER BY client_id;", connection=connection)


def update_payment_invoice(
    connection: sqlite3.Connection,
    new_payment_id: int,
    new_client_id: int,
    new_invoice_id: int,
    new_date: str,
    new_amount: float,
    new_payment_method: int,
) -> None:
    """Atomically record a payment and update the parent invoice balance."""
    with connection:
        connection.execute(
            """
            INSERT INTO payments (payment_id, client_id, invoice_id, date, amount, payment_method)
            VALUES (?, ?, ?, ?, ?, ?);
            """,
            (
                new_payment_id,
                new_client_id,
                new_invoice_id,
                new_date,
                new_amount,
                new_payment_method,
            ),
        )
        connection.execute(
            """
            UPDATE invoices
            SET payment_total = ROUND(payment_total + ?, 2)
            WHERE invoice_id = ?;
            """,
            (new_amount, new_invoice_id),
        )


refreshed = create_temp_invoice(conn)
assert len(refreshed) == 6

# Test atomic payment update on an isolated connection (initial payment_total = 100.00, invoice_total = 210.00)
tx_conn = sqlite3.connect(":memory:")
tx_conn.execute("PRAGMA foreign_keys = ON;")
tx_conn.executescript(SEED_PATH.read_text(encoding="utf-8"))

update_payment_invoice(tx_conn, 9, 6, 10, "2026-11-26", 110.00, 4)
inv10 = query(
    "SELECT invoice_total, payment_total FROM invoices WHERE invoice_id = 10;", connection=tx_conn
)[0]
print(f"Invoice 10 after payment: total={inv10['invoice_total']}, paid={inv10['payment_total']}")
assert inv10["payment_total"] == 210.00

# Verify rollback safety on invalid foreign key
failed = False
try:
    update_payment_invoice(tx_conn, 10, 9999, 10, "2026-11-26", 50.00, 1)
except sqlite3.IntegrityError as exc:
    failed = True
    print(f"Transaction aborted cleanly on invalid FK: {exc}")
assert failed
tx_conn.close()

In [ ]:
def calculate_office_salaries_cursor(connection: sqlite3.Connection) -> list[dict]:
    """Simulate a MySQL CURSOR loop building temporary table office_salaries."""
    connection.executescript("""
    DROP TABLE IF EXISTS office_salaries;
    CREATE TEMPORARY TABLE office_salaries (
        office_id INTEGER,
        total_salary REAL
    );
    """)
    office_ids = [
        row[0] for row in connection.execute("SELECT office_id FROM offices ORDER BY office_id;")
    ]
    for office_id in office_ids:
        total_salary = connection.execute(
            "SELECT COALESCE(SUM(salary), 0) FROM employees WHERE office_id = ?;",
            (office_id,),
        ).fetchone()[0]
        connection.execute(
            "INSERT INTO office_salaries (office_id, total_salary) VALUES (?, ?);",
            (office_id, total_salary),
        )
    return query("SELECT * FROM office_salaries ORDER BY office_id;", connection=connection)


cursor_res = calculate_office_salaries_cursor(conn)
set_based_res = query("""
SELECT
    o.office_id,
    CAST(COALESCE(SUM(e.salary), 0) AS REAL) AS total_salary
FROM offices o
LEFT JOIN employees e
    ON o.office_id = e.office_id
GROUP BY o.office_id
ORDER BY o.office_id;
""")

show_rows(cursor_res)
assert cursor_res == set_based_res
assert cursor_res[0]["total_salary"] == 538000.0